# 04 — Training Telemetry and Object Detection Metrics

This notebook analyzes training telemetry curves, derives the detection matching algorithm, and computes Precision, Recall, and Mean Average Precision (mAP) from first principles.

---

## 1. Ultralytics Execution Paradigms

Ultralytics YOLOv8 supports two primary workflows:

### CLI Execution
```bash
# Training
yolo task=detect mode=train model=yolov8s.pt data=./human_detection_mini/data.yaml epochs=20 imgsz=640 batch=16

# Validation
yolo task=detect mode=val model=./runs/detect/train/weights/best.pt data=./human_detection_mini/data.yaml
```

### Python SDK
```python
from ultralytics import YOLO

# Load model (pretrained weights)
model = YOLO("yolov8s.pt")

# Train model
results = model.train(data="./human_detection_mini/data.yaml", epochs=20, imgsz=640)

# Evaluate model
metrics = model.val()
```

In [ ]:
import csv
import json
from pathlib import Path

import numpy as np

DATA_DIR = Path("../data")
telemetry_path = DATA_DIR / "training_telemetry.csv"
benchmark_path = DATA_DIR / "detection_benchmark.json"

assert telemetry_path.exists()
assert benchmark_path.exists()
print("Telemetry logs and evaluation benchmarks located.")

## 2. Training Convergence & Loss Analysis

During training, three complementary loss components are logged across both training and validation splits:
- `box_loss`: Complete IoU loss evaluating bounding box geometry
- `cls_loss`: Binary Cross-Entropy loss evaluating classification correctness
- `dfl_loss`: Distribution Focal Loss evaluating coordinate boundary sharpness

In [ ]:
# Ingest telemetry rows
telemetry_records = []
with telemetry_path.open(encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        telemetry_records.append({k: float(v) for k, v in row.items()})

epochs = [int(r["epoch"]) for r in telemetry_records]
train_box = [r["train/box_loss"] for r in telemetry_records]
val_box = [r["val/box_loss"] for r in telemetry_records]
map50 = [r["metrics/mAP50(B)"] for r in telemetry_records]
map50_95 = [r["metrics/mAP50-95(B)"] for r in telemetry_records]

print(f"Total epochs logged: {len(epochs)}")
print(f"Initial train box loss: {train_box[0]:.4f} -> Final: {train_box[-1]:.4f}")
print(f"Initial val box loss:   {val_box[0]:.4f} -> Final: {val_box[-1]:.4f}")
print(f"Final mAP@0.50:         {map50[-1]:.4f}")
print(f"Final mAP@0.50:0.95:    {map50_95[-1]:.4f}")

# Ascertain monotonic convergence trend
assert train_box[-1] < train_box[0]
assert map50[-1] > map50[0]

## 3. Ground-Truth Proposal Matching & Metric Calculation

To evaluate a trained object detector on unseen images:
1. Candidate predictions are sorted descending by confidence score.
2. For each prediction, compute IoU against all ground truth boxes in that image.
3. If the maximum IoU $\ge \alpha$ (e.g., $\alpha = 0.50$) and that ground truth box has not yet been matched:
   - Mark prediction as **True Positive (TP)**
   - Mark that ground truth box as matched
4. Otherwise:
   - Mark prediction as **False Positive (FP)**
5. Any ground truth boxes remaining unmatched are counted as **False Negatives (FN)**.

$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}, \quad \text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}} = \frac{\text{TP}}{\text{Total Ground Truths}}$$

In [ ]:
def compute_box_iou(box_a: list[float], box_b: list[float]) -> float:
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def evaluate_detection_split(benchmark_file: Path, iou_thresh: float = 0.50) -> dict[str, float]:
    with benchmark_file.open(encoding="utf-8") as f:
        bench = json.load(f)

    all_predictions = []
    total_ground_truths = 0

    for sample in bench["samples"]:
        img_name = sample["image_name"]
        gts = sample["ground_truths"]
        total_ground_truths += len(gts)

        for p in sample["predictions"]:
            all_predictions.append(
                {
                    "img": img_name,
                    "confidence": p["confidence"],
                    "bbox": p["bbox_voc"],
                    "gts": gts,
                }
            )

    # Sort all predictions globally by confidence descending
    all_predictions.sort(key=lambda x: x["confidence"], reverse=True)

    matched_gt_ids = set()
    tp_list = []
    fp_list = []

    for pred in all_predictions:
        best_iou = 0.0
        best_gt_id = None
        for gt in pred["gts"]:
            iou = compute_box_iou(pred["bbox"], gt["bbox_voc"])
            if iou > best_iou:
                best_iou = iou
                best_gt_id = gt["id"]

        if best_iou >= iou_thresh and best_gt_id not in matched_gt_ids:
            tp_list.append(1)
            fp_list.append(0)
            matched_gt_ids.add(best_gt_id)
        else:
            tp_list.append(0)
            fp_list.append(1)

    tp_cumsum = np.cumsum(tp_list)
    fp_cumsum = np.cumsum(fp_list)

    recalls = tp_cumsum / total_ground_truths
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)

    # 11-point interpolated Average Precision (AP)
    ap_11 = 0.0
    for r_thresh in np.linspace(0.0, 1.0, 11):
        prec_at_r = precisions[recalls >= r_thresh]
        p_interp = np.max(prec_at_r) if len(prec_at_r) > 0 else 0.0
        ap_11 += p_interp / 11.0

    return {
        "tp": int(tp_cumsum[-1]),
        "fp": int(fp_cumsum[-1]),
        "fn": total_ground_truths - int(tp_cumsum[-1]),
        "precision": float(precisions[-1]),
        "recall": float(recalls[-1]),
        "ap_11_point": float(ap_11),
    }


eval_results = evaluate_detection_split(benchmark_path, iou_thresh=0.50)
print("Validation Split Evaluation Metrics (IoU >= 0.50):")
for metric_name, val in eval_results.items():
    print(f"  {metric_name}: {val:.4f}" if isinstance(val, float) else f"  {metric_name}: {val}")

assert eval_results["tp"] == 6
assert eval_results["fn"] == 0
assert eval_results["precision"] > 0.85
assert eval_results["recall"] == 1.0